In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (SparkSession.builder
         .appName("fraud_model")
         .master("local[*]")
         .config("spark.driver.memory", "4g")            
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.parquet.compression.codec", "snappy")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 11:23:21 WARN Utils: Your hostname, DESKTOP-NFCGTHI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 11:23:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/vishwa/MAST30034_Project_2/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/11 11:23:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/11 11:23:26 WARN Utils: Service 

4.2.0


In [ ]:
import shutil
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error

ROOT   = Path.cwd()
TABLES = ROOT / "tables"
OUT    = ROOT / "data" / "curated"      
FINAL  = ROOT / "data" / "final"      
OUT.mkdir(parents=True, exist_ok=True); FINAL.mkdir(parents=True, exist_ok=True)

DAYS = sorted(TABLES.glob("part_*/order_datetime=*"), key=lambda p: p.name)
print(len(DAYS), "day folders found")

def read_day(folder):
    tx = pd.read_parquet(folder, columns=["user_id", "merchant_abn", "order_id", "dollar_value"])
    return tx[tx["dollar_value"] >= 0.01]

def day_str(folder):
    return folder.name.split("=")[1]

def daily_feats(tx, key, other, d):
    g = tx.groupby(key).agg(n_txn=("dollar_value", "size"),
                            total_value=("dollar_value", "sum"),
                            avg_value=("dollar_value", "mean"),
                            max_value=("dollar_value", "max"),
                            n_other=(other, "nunique")).reset_index()
    g["dow"] = pd.Timestamp(d).dayofweek
    return g

def load_labels(file, key):
    lab = pd.read_csv(TABLES / "part_1" / file)
    lab["date"] = pd.to_datetime(lab["order_datetime"]).dt.strftime("%Y-%m-%d")
    return lab.groupby([key, "date"], as_index=False)["fraud_probability"].mean()

cons_lab = load_labels("consumer_fraud_probability.csv", "user_id")
merc_lab = load_labels("merchant_fraud_probability.csv", "merchant_abn")
print(len(cons_lab), "consumer label rows |", len(merc_lab), "merchant label rows")

606 day folders found
34765 consumer label rows | 114 merchant label rows


In [ ]:
cons_by_day = {d: g for d, g in cons_lab.groupby("date")}
cons_train, merc_parts = [], []

for i, folder in enumerate(DAYS):
    d = day_str(folder)
    tx = read_day(folder)
    if d in cons_by_day:                                  
        cf = daily_feats(tx, "user_id", "merchant_abn", d)
        cons_train.append(cf.merge(cons_by_day[d], on="user_id", how="inner"))
    merc_parts.append(tx.groupby("merchant_abn").agg(
        total_value=("dollar_value", "sum"),
        n_other=("user_id", "nunique")).reset_index())
    if i % 100 == 0: print(i, "/", len(DAYS))

cons_train = pd.concat(cons_train, ignore_index=True)
merc_all   = pd.concat(merc_parts, ignore_index=True)

merc_feat = merc_all.groupby("merchant_abn").agg(
    avg_daily_value=("total_value", "mean"),
    avg_daily_customers=("n_other", "mean"),
    active_days=("total_value", "size")).reset_index()
merc_known = merc_lab.groupby("merchant_abn", as_index=False)["fraud_probability"].mean()
merc_train = merc_feat.merge(merc_known, on="merchant_abn")

print("consumer labelled days:", len(cons_train), "| merchants with a label:", len(merc_train))

0 / 606
100 / 606
200 / 606
300 / 606
400 / 606
500 / 606
600 / 606
consumer labelled days: 34765 | merchants with a label: 61


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

LOGF = ["n_txn", "total_value", "avg_value", "max_value", "n_other"]
SIZE = ["avg_daily_value", "avg_daily_customers", "active_days"]

def prep(df):
    X = np.log1p(df[LOGF].clip(lower=0))
    X["dow"] = df["dow"].values
    return X

def ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))

results = []
def score(y_true, y_pred, y_base, title, n_train, n_test):
    r = dict(test=title, n_train=n_train, n_test=n_test,
             MAE=round(mean_absolute_error(y_true, y_pred), 2),
             baseline_MAE=round(mean_absolute_error(y_true, y_base), 2),
             RMSE=round(mean_squared_error(y_true, y_pred) ** 0.5, 2),
             baseline_RMSE=round(mean_squared_error(y_true, y_base) ** 0.5, 2),
             R2=round(r2_score(y_true, y_pred), 3),
             spearman=round(spearmanr(y_true, y_pred)[0], 2))
    results.append(r); print(r)

def report(tr, te, title):
    y, yt = tr["fraud_probability"], te["fraud_probability"]
    m = ridge().fit(prep(tr), y)
    p = np.clip(m.predict(prep(te)), y.min(), y.max())
    score(yt, p, np.full(len(te), y.mean()), title, len(tr), len(te))

#consumer: random 80/20
d = cons_train.sample(frac=1, random_state=42); k = int(len(d) * 0.8)
report(d.iloc[:k], d.iloc[k:], "CONSUMER random 80/20")


tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                  .split(cons_train, groups=cons_train["user_id"]))
report(cons_train.iloc[tr_i], cons_train.iloc[te_i], "CONSUMER 80/20 split by user")

d = cons_train.sort_values("total_value"); k = int(len(d) * 0.8)
report(d.iloc[:k], d.iloc[k:], "CONSUMER spend split (train low, test high)")


Xm, ym = np.log1p(merc_train[SIZE]), merc_train["fraud_probability"]
pm = np.clip(cross_val_predict(ridge(), Xm, ym, cv=LeaveOneOut()), ym.min(), ym.max())
base = (ym.sum() - ym) / (len(ym) - 1)
score(ym, pm, base, "MERCHANT leave-one-out", len(ym) - 1, 1)

ev = pd.DataFrame(results)
ev.to_csv(FINAL / "model_evaluation.csv", index=False)
print(ev.T)

{'test': 'CONSUMER random 80/20', 'n_train': 27812, 'n_test': 6953, 'MAE': 2.79, 'baseline_MAE': 5.84, 'RMSE': 4.71, 'baseline_RMSE': 9.28, 'R2': 0.742, 'spearman': np.float64(0.81)}
{'test': 'CONSUMER 80/20 split by user', 'n_train': 27730, 'n_test': 7035, 'MAE': 2.83, 'baseline_MAE': 5.95, 'RMSE': 4.73, 'baseline_RMSE': 9.41, 'R2': 0.747, 'spearman': np.float64(0.8)}
{'test': 'CONSUMER spend split (train low, test high)', 'n_train': 27812, 'n_test': 6953, 'MAE': 9.81, 'baseline_MAE': 16.02, 'RMSE': 13.86, 'baseline_RMSE': 21.28, 'R2': 0.05, 'spearman': np.float64(0.75)}
{'test': 'MERCHANT leave-one-out', 'n_train': 60, 'n_test': 1, 'MAE': 10.85, 'baseline_MAE': 18.04, 'RMSE': 15.69, 'baseline_RMSE': 20.29, 'R2': 0.381, 'spearman': np.float64(0.64)}
                                   0                             1  \
test           CONSUMER random 80/20  CONSUMER 80/20 split by user   
n_train                        27812                         27730   
n_test                       

In [6]:
cons_model = ridge().fit(prep(cons_train), cons_train["fraud_probability"])
SPEND_MIN = cons_train["total_value"].min()

merc_model = ridge().fit(Xm, ym)
M_LO, M_HI = ym.min(), ym.max()
merc_feat["predicted_merchant_fraud_probability"] = np.clip(
    merc_model.predict(np.log1p(merc_feat[SIZE])), M_LO, M_HI)

rng = merc_train["avg_daily_value"]
merc_feat["outside_labelled_size_range"] = ~merc_feat["avg_daily_value"].between(rng.min(), rng.max())
merc_feat = merc_feat.merge(
    merc_known.rename(columns={"fraud_probability": "known_merchant_label"}),
    on="merchant_abn", how="left")
merc_feat["merchant_fraud_probability_final"] = merc_feat["known_merchant_label"].fillna(
    merc_feat["predicted_merchant_fraud_probability"])
merc_feat["merchant_risk_percentile"] = merc_feat["predicted_merchant_fraud_probability"].rank(pct=True) * 100

print(merc_feat["predicted_merchant_fraud_probability"].describe().round(2))
print("merchants outside labelled size range:", merc_feat["outside_labelled_size_range"].mean().round(3))

count    4422.00
mean       56.58
std         8.39
min        18.47
25%        51.67
50%        57.17
75%        62.58
max        74.63
Name: predicted_merchant_fraud_probability, dtype: float64
merchants outside labelled size range: 0.004


In [ ]:
SCORED = OUT / "scored"
shutil.rmtree(SCORED, ignore_errors=True); SCORED.mkdir()
mlook = merc_feat[["merchant_abn", "merchant_fraud_probability_final",
                   "merchant_risk_percentile", "outside_labelled_size_range"]]

for i, folder in enumerate(DAYS):
    if folder.parent.name == "part_1":
        continue
    d = day_str(folder)
    tx = read_day(folder)
    cf = daily_feats(tx, "user_id", "merchant_abn", d)
    cf["consumer_raw_score"] = cons_model.predict(prep(cf))          
    cf["in_labelled_range"] = cf["total_value"] >= SPEND_MIN
    if d in cons_by_day:
        cf = cf.merge(cons_by_day[d].rename(columns={"fraud_probability": "consumer_known_label"}),
                      on="user_id", how="left")
    else:
        cf["consumer_known_label"] = np.nan
    out = (tx.merge(cf[["user_id", "consumer_raw_score", "consumer_known_label", "in_labelled_range"]],
                    on="user_id", how="left")
             .merge(mlook, on="merchant_abn", how="left"))
    out["order_datetime"] = d
    out.to_parquet(SCORED / f"{d}.parquet")
    if i % 100 == 0: print(i, "/", len(DAYS))
print("done")

0 / 606
100 / 606
200 / 606
300 / 606
400 / 606
500 / 606
600 / 606
done


In [8]:
files = sorted(SCORED.glob("*.parquet"))

samp = []
for f in files:
    s = pd.read_parquet(f, columns=["consumer_raw_score"])["consumer_raw_score"].dropna()
    samp.append(s.sample(min(1000, len(s)), random_state=0))
GRID = np.quantile(pd.concat(samp), np.linspace(0, 1, 1001))
PCT  = np.linspace(0, 100, 1001)

for f in files:
    df = pd.read_parquet(f)
    df["consumer_risk_percentile"] = np.interp(df["consumer_raw_score"], GRID, PCT)
    df["risk_score"] = 0.5 * df["consumer_risk_percentile"] + 0.5 * df["merchant_risk_percentile"]
    df.to_parquet(f)
print("done")

done


In [ ]:
COLS = ["order_id", "user_id", "merchant_abn", "order_datetime", "dollar_value",
        "consumer_risk_percentile", "merchant_risk_percentile", "risk_score",
        "consumer_known_label", "merchant_fraud_probability_final",
        "in_labelled_range", "outside_labelled_size_range"]

writer, schema, merch_parts, n = None, None, [], 0
for f in files:
    df = pd.read_parquet(f)
    t = df[COLS]
    tbl = pa.Table.from_pandas(t, preserve_index=False) if schema is None \
          else pa.Table.from_pandas(t, schema=schema, preserve_index=False)
    if writer is None:
        schema = tbl.schema
        writer = pq.ParquetWriter(FINAL / "transaction_risk_scores.parquet", schema)
    writer.write_table(tbl)
    n += len(df)
    merch_parts.append(df.groupby("merchant_abn").agg(
        n_txn=("risk_score", "size"), sales=("dollar_value", "sum"),
        risk_sum=("risk_score", "sum")).reset_index())
writer.close()


ms = pd.concat(merch_parts).groupby("merchant_abn", as_index=False).sum()
ms["avg_risk_score"] = ms["risk_sum"] / ms["n_txn"]
ms = ms.drop(columns="risk_sum").merge(
    merc_feat[["merchant_abn", "merchant_fraud_probability_final", "merchant_risk_percentile",
               "known_merchant_label", "outside_labelled_size_range"]], on="merchant_abn", how="left")

ms["risk_tier"] = pd.cut(ms["merchant_risk_percentile"], [-0.1, 50, 90, 100.1],
                         labels=["low", "medium", "high"])
ms.sort_values("avg_risk_score", ascending=False).to_csv(FINAL / "merchant_risk_summary.csv", index=False)

print(f"{n:,} transactions written to {FINAL/'transaction_risk_scores.parquet'}")
print(f"{len(ms):,} merchants written to {FINAL/'merchant_risk_summary.csv'}")
print(ms["risk_tier"].value_counts())

14,194,481 transactions written to /home/vishwa/MAST30034_Project_2/data/final/transaction_risk_scores.parquet
4,422 merchants written to /home/vishwa/MAST30034_Project_2/data/final/merchant_risk_summary.csv
risk_tier
low       2211
medium    1768
high       443
Name: count, dtype: int64


In [ ]:

samp = pd.concat(pd.read_parquet(f, columns=["consumer_risk_percentile", "risk_score", "in_labelled_range"])
                   .sample(500, random_state=0) for f in files[:: max(1, len(files)//300)])
print(samp[["consumer_risk_percentile", "risk_score"]].describe().round(1))
print("share of sampled rows in labelled spend range:", samp["in_labelled_range"].mean().round(3))

all_days = pd.concat(read_day(f).groupby("user_id")["dollar_value"].sum() for f in DAYS[::10])
q = [.05, .25, .5, .75, .95]
spend = pd.DataFrame({"all consumer-days": all_days.quantile(q).round(0),
                      "labelled days": cons_train["total_value"].quantile(q).round(0)})
spend.index = ["5th", "25th", "median", "75th", "95th"]
spend.to_csv(FINAL / "spend_comparison.csv")
print(spend)
print("share of all days at or above the smallest labelled day:", (all_days >= SPEND_MIN).mean().round(3))


print(pd.read_parquet(files[100], columns=COLS).sample(5, random_state=1).round(2).T)

       consumer_risk_percentile  risk_score
count                  151500.0    151500.0
mean                       50.1        32.4
std                        28.9        19.2
min                         0.0         0.0
25%                        25.1        16.8
50%                        49.9        31.4
75%                        75.1        45.7
max                       100.0        99.3
share of sampled rows in labelled spend range: 0.036
        all consumer-days  labelled days
5th                   9.0         2077.0
25th                 43.0         2630.0
median              110.0         3467.0
75th                269.0         5129.0
95th                911.0        12767.0
share of all days at or above the smallest labelled day: 0.026
                                                                 18865  \
order_id                          b296522d-18f9-4f24-a8dc-62d7d3255679   
user_id                                                          16686   
merchant_abn        